# Select the fingerprint decision threshold on the **validation** split

Every `predict.py` in this repository binarises the predicted fingerprint at a fixed 0.5. This notebook sweeps
the threshold on `val_results.pkl`, keeps the argmax, and applies that single value to the test split.

**Prerequisite** — validation predictions. Produce them with:

```bash
cd ../mist            && python predict.py --checkpoint <run folder> --split val
cd ../other_baselines && python predict.py --checkpoint <run folder> --split val
```

The logic lives in `benchmarked_models/tune_threshold.py` so that the same selection is applied to MIST, the
trained baselines and the retrieval baselines.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))          # benchmarked_models/
import tune_threshold as tt

## Run folders

Point this at the run folders to tune. Each must contain `val_results.pkl` and `test_results.pkl`.

In [ ]:
RESULTS = Path("../../results/FP_prediction")

run_folders = sorted(p for p in RESULTS.glob("*/*/*") if p.is_dir())
print(f"{len(run_folders)} run folders")
for p in run_folders[:10]:
    print("  ", p.relative_to(RESULTS))

## Sweep on validation, report on test

`tt.tune` writes `test_performance_val_threshold.json` into each run folder. `pred` may hold logits
(`other_baselines`) or probabilities (MIST, nearest neighbour); `scale="auto"` tells them apart.

In [ ]:
GRID = tt.DEFAULT_GRID      # 0.05 .. 0.975 step 0.025

for run in run_folders:
    print(f"{run.relative_to(RESULTS)}: {tt.tune(run, GRID, scale='auto', overwrite=True)}")

## Summary

`threshold` was chosen on validation.

In [ ]:
rows = []
for run in run_folders:
    f = run / "test_performance_val_threshold.json"
    if not f.exists():
        continue
    r = json.load(open(f))
    rows.append((run.parts[-3], run.name, r["threshold"], r["val_jaccard"],
                 r["test_jaccard"], r["test_jaccard_at_0.5"], r["test_jaccard"] - r["test_jaccard_at_0.5"]))

if rows:
    import pandas as pd
    df = pd.DataFrame(rows, columns=["model", "run", "threshold", "val J", "test J", "test @ 0.5", "delta"])
    print(df.to_string(index=False))
else:
    print("No run folder has val_results.pkl yet -- see the prerequisite above.")